# Storing and Retrieving Data with Memory in LangChain and AWS

## 1. Theory & Real-World Context

Large Language Models (LLMs) are stateless by design; they evaluate each prompt in isolation. To build conversational AI, the application infrastructure must act as the model's memory, injecting the history of the conversation into every new prompt.

While your transcript introduces `ConversationBufferMemory` and `ConversationChain`, these are **legacy, in-memory components** designed for prototyping. In a production environment, if your application container restarts, all in-memory chat history is permanently lost.

**The Modern AWS Production Standard:**
Modern AI engineering relies on **LangChain Expression Language (LCEL)** wrappers like `RunnableWithMessageHistory` paired with an external, persistent database—most commonly **Amazon DynamoDB** via `DynamoDBChatMessageHistory`. This architecture allows your AI agents to scale horizontally; a user can send a message, hit any server in your backend fleet, and the system instantly pulls their conversation history from DynamoDB before querying Amazon Bedrock.

---

## 2. Visual Architecture

```mermaid
flowchart TD
    subgraph Client ["Client Application"]
        User[User Prompt: 'What is my name?']
        Session[Session ID: 'user_123']
    end

    subgraph LCEL ["LangChain Expression Language (LCEL)"]
        direction TB
        RWH[RunnableWithMessageHistory<br/>State Manager]
        Prompt[ChatPromptTemplate<br/>w/ MessagesPlaceholder]
        LLM[ChatBedrockConverse]
    end

    subgraph AWSCloud ["AWS Cloud"]
        DDB[(Amazon DynamoDB<br/>ChatMessageHistory)]
        BR[Amazon Bedrock<br/>Claude/Nova]
    end

    User --> RWH
    Session --> RWH
    RWH <--> |"1. Fetch past messages<br/>4. Save new turn"| DDB
    RWH -- "2. Inject context" --> Prompt
    Prompt --> LLM
    LLM <--> |"3. Inference API"| BR

```

---

## 3. Console Hands-On

**Objective:** Provision a serverless Amazon DynamoDB table to persist LangChain chat sessions across application restarts.

1. Navigate to the **Amazon DynamoDB Console** $\rightarrow$ Click **Create table**.
2. **Table name:** `LangChainSessionHistory`.
3. **Partition key (Primary Key):** Type `SessionId` and select **String**. *(Note: LangChain's `DynamoDBChatMessageHistory` defaults to looking for a primary key exactly named `SessionId`).*
4. Leave Sort key blank.
5. Under **Table settings**, select **Customize settings**.
6. Under **Read/write capacity**, select **On-demand** (ideal for unpredictable chatbot traffic).
7. Click **Create table**. This table will now serve as the centralized memory store for your entire AI fleet.

---

## 4. Boto3 & LangChain Implementation

This script demonstrates the modern LCEL approach to memory, moving away from legacy `ConversationChain` into highly scalable `RunnableWithMessageHistory` backed by DynamoDB.

*Prerequisites: `pip install langchain-aws langchain-community boto3*`

```python
import logging
import boto3
from botocore.exceptions import ClientError
from langchain_aws import ChatBedrockConverse
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import DynamoDBChatMessageHistory

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("dynamo_memory_manager")

class StatefulChatManager:
    """Manages persistent chat memory using DynamoDB and Bedrock."""

    def __init__(self, table_name: str, region_name: str = "us-east-1"):
        self.table_name = table_name
        self.boto3_session = boto3.Session(region_name=region_name)
        
        # 1. Initialize modern ChatBedrockConverse
        self.llm = ChatBedrockConverse(
            client=self.boto3_session.client("bedrock-runtime"),
            model="anthropic.claude-3-5-sonnet-20241022-v2:0",
            temperature=0.3
        )
        
        # 2. Define prompt with a MessagesPlaceholder for injected history
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a concise AI travel assistant."),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{question}")
        ])
        
        # 3. Create the stateless LCEL chain
        self.chain = self.prompt | self.llm
        
        # 4. Wrap the chain with the State Manager
        self.chain_with_history = RunnableWithMessageHistory(
            runnable=self.chain,
            get_session_history=self._get_dynamodb_history,
            input_messages_key="question",
            history_messages_key="chat_history"
        )

    def _get_dynamodb_history(self, session_id: str) -> DynamoDBChatMessageHistory:
        """Factory function that retrieves/creates a DynamoDB history object per session."""
        return DynamoDBChatMessageHistory(
            table_name=self.table_name,
            session_id=session_id,
            boto3_session=self.boto3_session
        )

    def chat(self, session_id: str, user_input: str) -> str:
        """Invokes the chain, automatically fetching and updating DynamoDB history."""
        try:
            logger.info(f"Invoking chat for session: {session_id}")
            
            # The config dict passes the session_id to the factory function
            config = {"configurable": {"session_id": session_id}}
            
            response = self.chain_with_history.invoke(
                {"question": user_input},
                config=config
            )
            return response.content
            
        except ClientError as e:
            logger.error(f"AWS Error: {e.response['Error']['Message']}")
            return "Connection error."
        except Exception as e:
            logger.error(f"Execution Error: {str(e)}")
            return "Application error."


if __name__ == "__main__":
    TABLE_NAME = "LangChainSessionHistory"
    chat_manager = StatefulChatManager(table_name=TABLE_NAME)
    
    # Simulate User Turn 1
    session = "user_abc123"
    print("User: Hi, I'm visiting Los Angeles.")
    r1 = chat_manager.chat(session, "Hi, I'm visiting Los Angeles.")
    print(f"AI: {r1}\n")
    
    # Simulate User Turn 2 - The AI will implicitly know the location
    print("User: What is the closest beach?")
    r2 = chat_manager.chat(session, "What is the closest beach?")
    print(f"AI: {r2}\n")

```

---

## 5. Practice Challenges

### Challenge 1: The Context Window Exhaustion Trap

**Scenario:** A user chats with your bot for 3 hours. DynamoDB faithfully saves all 500 messages and injects them into the `chat_history` placeholder. The prompt becomes 250,000 tokens long, causing a massive AWS bill and an `HTTP 400 Context Window Exceeded` error.

* **Task:** Import `trim_messages` from `langchain_core.messages`. Update the LCEL chain to insert the trimmer directly between the `chat_history` placeholder and the LLM execution, configuring it to keep only the last `max_tokens=2048` worth of messages.

### Challenge 2: Conversation Summary Memory

**Scenario:** Instead of a strict cutoff (which causes the bot to immediately forget older constraints), you want the bot to maintain a condensed summary of the entire 3-hour chat.

* **Task:** Implement `ConversationSummaryBufferMemory`. Configure it to rely on a cheaper, faster model (`amazon.nova-micro-v1:0`) to silently run a background summarization task on the chat history every time the token count exceeds 1,000 tokens, preserving the core themes while flushing the raw text.

---

## 6. Industry Standards & Tips

* **Avoid Local Memory:** Never use `ConversationBufferMemory` or `ConversationChain` in Dockerized, ECS, or Lambda environments. Every time the container spins down, your users' chat history resets. `DynamoDBChatMessageHistory` is the AWS gold standard because it decouples state from compute.
* **Data Lifecycle Management:** Unchecked chat logs cost money. Always configure **Time to Live (TTL)** on your DynamoDB table. Set LangChain to write a `ttl` attribute (e.g., 30 days from creation) so AWS automatically deletes stale chat sessions, saving storage costs.
* **Multi-Tenancy Security:** When building SaaS applications, never use a generic session ID like `session_1`. Construct composite keys like `tenant_id#user_id#thread_id` to ensure absolute isolation between different customers chatting simultaneously.

---

## 7. Interview Q&A (Targeted for AI Engineers)

### Q1: Why do Large Language Models need "memory" components? Why can't they just remember what we talked about?

> **Answer:** Large Language Models are inherently stateless functions. An API request to Amazon Bedrock has no awareness of the request sent 5 seconds prior. To simulate conversation, the application layer must retrieve the entire historical transcript and prepend it to the new prompt. "Memory" in AI engineering refers to the application's state management, not the model's brain.

### Q2: You inherited a LangChain prototype using `ConversationChain` and `ConversationBufferMemory`. You need to deploy this to AWS Lambda. What architectural changes are required?

> **Answer:** First, I would refactor the code to drop `ConversationChain` in favor of LangChain Expression Language (LCEL) with `RunnableWithMessageHistory` to align with modern best practices. Second, because AWS Lambda is ephemeral compute, `ConversationBufferMemory` will wipe on every cold start. I must replace it with a persistent storage backend like `DynamoDBChatMessageHistory` so state survives across isolated Lambda invocations.

### Q3: How do you prevent a highly active user from exhausting a model's token limit during a long chat session?

> **Answer:** If we dump the entire DynamoDB transcript into every prompt, we will eventually hit a context window limit (and incur massive per-token costs). We implement memory management strategies: a **Sliding Window** (e.g., passing only the last 10 messages using `trim_messages`), or a **Summary Buffer** (using a cheaper LLM to continuously compress older messages into a short summary paragraph while keeping the most recent messages verbatim).